# Typing, dataclasses, Protocols

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### Typing

> **Problem.** A refactor renames `content` to `text` in the message dictionaries. Forty call sites are updated; the forty-first, in a rarely used export job, still reads `message["content"]`. Tests pass. It fails on the first export in production, a month later.

**Idea.** Write the shape of data next to the code; a checker finds mismatches before the code runs.

**Use when** any code with more than one author or one week of life.  
**Not when** a throwaway script.

```mermaid
flowchart LR
    C[code + type hints] --> P[pyright in CI]
    P -->|clean| M[merge]
    P -->|"'moderator' not allowed"| B[build fails]
```

**How it works.**
1. `class Message(TypedDict)` declares the keys a message dictionary has and their types; `role: Literal["system", "user", "assistant"]` restricts the value to those three strings.
2. Functions declare what they take and return: `build_messages(system: str, user: str) -> list[Message]`.
3. Python ignores all of this at runtime — a wrong value still runs.
4. `pyright` reads the annotations and checks every assignment and call; in the cell it runs on a snippet with two deliberate bugs.
5. In a real repo it runs in CI on every pull request and fails the build on any error, so the forty-first call site is caught before merge.

| | what happens | result |
|:--|:--|:--|
| ✓ | `{"role": "user", …}` | passes |
| ✗ | `{"role": "moderator", …}` | pyright: not assignable to Literal |
| ✗ | `count: int = "3"` | pyright: str is not int |

**Production code and its real output**

In [2]:
# Typing — message shapes as TypedDicts, checked by pyright in CI. The snippet below contains two
# bugs a reviewer could miss; the type checker catches both before the code runs.
import subprocess
import tempfile
from pathlib import Path
from typing import Literal, TypedDict


class Message(TypedDict):
    role: Literal["system", "user", "assistant"]
    content: str


def build_messages(system: str, user: str) -> list[Message]:
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


print(build_messages("Be brief.", "Explain typing."))

snippet = """
from typing import Literal, TypedDict
class Message(TypedDict):
    role: Literal["system", "user", "assistant"]
    content: str
bad: Message = {"role": "moderator", "content": "hi"}   # invalid role
count: int = "3"                                        # wrong type
"""
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "snippet.py"
    path.write_text(snippet)
    report = subprocess.run(
        ["uv", "run", "pyright", "--outputjson", str(path)], capture_output=True, text=True
    )
diagnostics = json.loads(report.stdout)["generalDiagnostics"]
for item in diagnostics:
    print(
        
            f"pyright line {item['range']['start']['line'] + 1}: "
            f"{item['message'].splitlines()[0][:80]}"
        
    )
assert len(diagnostics) == 2

[{'role': 'system', 'content': 'Be brief.'}, {'role': 'user', 'content': 'Explain typing.'}]


pyright line 6: Type "dict[str, str]" is not assignable to declared type "Message"
pyright line 7: Type "Literal['3']" is not assignable to declared type "int"


**What the output shows.** pyright reported exactly the two planted bugs with line numbers — a wrong `Literal` value and a string assigned to an `int` — without running the code.

**In practice**
- **type the edges** — provider JSON is untyped; parse it into a model at the boundary so everything inside is typed and the checker can help.
- **no Any** — `Any` turns the checker off for that value and everything derived from it — the mistake hides exactly where you needed the check.
- **CI with a threshold** — a checker that only prints warnings is decoration; fail the build.
- **Protocol for interfaces** — type the capability you need (`ChatModel`), not the vendor class, or the types lock you to one SDK.
- **incremental adoption** — on an existing codebase, enable checking file by file; a flood of 2,000 errors on day one gets the tool switched off.

**Alternatives** — runtime checks only (Pydantic everywhere — safer, slower) · no types (fast to write, slow to debug)

**Terms** — *type hint*: `name: str` — a label the checker reads, Python ignores · *TypedDict*: the keys and value types of a dictionary · *Literal*: only these exact values


### dataclasses

> **Problem.** A cost-tracking function builds a `Usage` record for every one of 50,000 requests an hour. Someone made `Usage` a Pydantic model "to be safe"; every construction re-validates two integers that came from your own code. Profiling shows validation at 8% of CPU for data that was never untrusted.

**Idea.** A plain typed record for data you already trust.

**Use when** internal value objects: prices, usage, results.  
**Not when** data from users or models — nothing is checked.

```
                Pydantic              dataclass
checks data     yes                   no
speed           slower                fast
use for         data from outside     data inside the program
```

**How it works.**
1. `@dataclass` above a class generates `__init__`, `__eq__` and `__repr__` from the field list; you write only the fields.
2. `frozen=True` makes instances immutable: assigning to a field raises `FrozenInstanceError`, and the object becomes hashable (usable as a dict key).
3. `field(default_factory=list)` creates a fresh list per instance; a bare `= []` would be one list shared by all instances.
4. Methods are ordinary: `cost_usd(self, price)` reads its own fields and the price table.
5. `asdict(record)` converts to a plain dictionary for logging or JSON.

| | what happens | result |
|:--|:--|:--|
| ✓ | `Price("gpt-4o-mini", 0.15) == Price("gpt-4o-mini", 0.15)` | True — equal by value |
| ✓ | `frozen=True`, then `p.usd = 1` | FrozenInstanceError — constants stay constant |

**Production code and its real output**

In [3]:
# Dataclasses — internal value objects with no I/O boundary (Pydantic at the edges, dataclasses
# inside). frozen=True for constants like a price table.
from dataclasses import asdict, dataclass, field


@dataclass(frozen=True)
class ModelPrice:
    model: str
    input_usd_per_million: float
    output_usd_per_million: float


@dataclass
class Usage:
    prompt_tokens: int
    completion_tokens: int

    def cost_usd(self, price: ModelPrice) -> float:
        return (
            self.prompt_tokens * price.input_usd_per_million
            + self.completion_tokens * price.output_usd_per_million
        ) / 1e6


@dataclass
class CompletionRecord:
    request_id: str
    usage: Usage
    tags: list[str] = field(default_factory=list)


price = ModelPrice("gpt-4o-mini", 0.15, 0.60)
record = CompletionRecord("req-1", Usage(1200, 300), tags=["chat"])
show("record", asdict(record))
print("cost usd:", round(record.usage.cost_usd(price), 6))
assert abs(record.usage.cost_usd(price) - 0.00036) < 1e-9

record
{
  "request_id": "req-1",
  "usage": {
    "prompt_tokens": 1200,
    "completion_tokens": 300
  },
  "tags": [
    "chat"
  ]
}
cost usd: 0.00036


**What the output shows.** The record printed as a clean nested dictionary and the cost came out to $0.00036 for 1,200 prompt + 300 completion tokens at gpt-4o-mini prices.

**In practice**
- **the rule** — Pydantic at the edges, dataclasses inside. Convert once at the boundary; pass dataclasses after that.
- **frozen for constants** — price tables, model configs, anything shared across threads — immutability removes a whole class of bugs.
- **slots** — `@dataclass(slots=True)` cuts memory per instance noticeably when you hold millions.
- **mutable defaults** — `= []` or `= {}` on a field is the classic bug: every object shares the same list.
- **no validation, by design** — if you find yourself adding checks in `__post_init__`, the data is not trusted — it should have been a Pydantic model.

**Alternatives** — Pydantic (validation) · `NamedTuple` (immutable, tuple-like, no methods needed) · plain dict (no names, no checks, no editor help)

**Terms** — *frozen*: cannot be changed after creation · *default_factory*: make a fresh list per object


### Protocols

> **Problem.** Business code calls `openai_client.chat.completions.create(...)` in 40 files. The company signs a contract with a second provider and wants to route 20% of traffic there. Every one of the 40 files needs an `if provider == ...`, and every unit test still hits the real OpenAI API because there is no seam to put a fake in.

**Idea.** Depend on what you need — a method signature — not on a vendor's class; adapters plug in, tests use a fake.

**Use when** a second implementation or a test double exists or is coming.  
**Not when** one implementation forever — an abstraction with one user is noise.

```mermaid
flowchart LR
    B[business code] --> P["ChatModel · complete(prompt) → str"]
    P --- O[OpenAI adapter]
    P --- L[Ollama adapter]
    P --- F[fake for tests]
```

**How it works.**
1. `class ChatModel(Protocol)` declares one method signature: `complete(self, prompt: str) -> str`. No implementation, no inheritance required.
2. `OpenAIChatModel` implements `complete` with the real SDK. `FakeChatModel` implements it by returning a canned string and recording the prompt.
3. Neither class mentions `ChatModel`; they satisfy it just by having the method (structural typing).
4. `summarise(model: ChatModel, text)` is written against the Protocol; the type checker accepts any object with that method.
5. Production passes the OpenAI adapter; tests pass the fake and assert on the captured prompt — no network, no cost.

| | what happens | result |
|:--|:--|:--|
| ✓ real | `summarise(OpenAIChatModel(), text)` | API call, real answer |
| ✓ fake | `summarise(FakeChatModel(…), text)` | no network, prompt captured for assertions |

**Production code and its real output**

In [4]:
# Protocols — the interface the application depends on. The real client and a test double both
# satisfy it structurally; business code never imports a vendor SDK.
from typing import Protocol


class ChatModel(Protocol):
    def complete(self, prompt: str) -> str: ...


class OpenAIChatModel:
    def complete(self, prompt: str) -> str:
        response = client.chat.completions.create(
            model=settings.openai_model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content or ""


class FakeChatModel:
    """Test double: returns a canned answer and records what it was asked."""

    def __init__(self, canned: str) -> None:
        self.canned, self.prompts = canned, []

    def complete(self, prompt: str) -> str:
        self.prompts.append(prompt)
        return self.canned


def summarise(model: ChatModel, text: str) -> str:
    return model.complete(f"Summarise in five words: {text}")


text = "The Pacific is the largest and deepest of Earth's five oceanic divisions."
print("real:", summarise(OpenAIChatModel(), text))
fake = FakeChatModel("Pacific is the largest ocean.")
print("fake:", summarise(fake, text), "| captured prompt:", fake.prompts[0][:32], "...")
assert fake.prompts[0].startswith("Summarise in five words")

real: Pacific: largest, deepest oceanic division.
fake: Pacific is the largest ocean. | captured prompt: Summarise in five words: The Pac ...


**What the output shows.** The same `summarise` function produced a real five-word summary through OpenAI and a canned one through the fake; the fake recorded the exact prompt it was given.

**In practice**
- **wait for the second** — introduce the Protocol when the second implementation or the test fake appears; before that it is speculation.
- **narrow** — one capability per Protocol (`ChatModel`, `Embedder`, `VectorStore`); a 15-method interface is just the vendor SDK in disguise.
- **contract tests** — run one shared test suite against the real adapter and the fake, or the fake drifts and tests pass against behaviour that no longer exists.
- **no leaks** — if the Protocol's arguments are OpenAI message dicts, you have not decoupled anything — define your own small types.
- **adapters own the mess** — retries, error mapping, token counting live in the adapter, so business code never sees a vendor exception.

**Alternatives** — abstract base classes (explicit inheritance, `@abstractmethod`) · duck typing with no declared interface (works, but the checker cannot help)

**Terms** — *Protocol*: an interface defined by method signatures · *adapter*: a small class translating your interface to a vendor SDK · *fake*: a stand-in returning canned answers
